# Benchmarking logical state-preparation circuits

Fault-tolerant quantum computations rely on code blocks prepared in logical states such as $|0_L\rangle$ and $|+_L\rangle$, and the quality of a noisy preparation circuit affects every later step that uses its output.
Preparation circuits are commonly benchmarked in two ways.
An exact certificate of fault tolerance, as in the [bounded-propagation example](../logical_error_rates/5_state_preparation.ipynb#Check-bounded-fault-propagation), checks that a few faults cannot leave a heavy error on the prepared block, but its yes-or-no answer cannot rank two circuits that both pass.
A logical error rate, as in the [flagged-preparation benchmark](../logical_error_rates/5_state_preparation.ipynb#Flagged-Steane-preparation), decodes the prepared block and estimates how often a logical operator that should stabilize the prepared state comes out wrong.

A logical error rate counts how often those logical operators fail after decoding.
For the $|0_L\rangle$ state of a code with one logical qubit, that operator is the logical $Z_L$, which a Z-type Pauli error cannot flip.
A Z-type error therefore cannot change the outcome of this benchmark by itself, even though the circuits that use the block next must still correct it; at most it can confuse the decoder about other errors, by an amount that depends on the decoder.
Likewise, an error that the decoder corrects does not count as a failure, even though it uses up part of the code's capacity to correct further errors in those circuits.

This notebook estimates a figure of merit that accounts for both kinds of error.
It counts errors of both Pauli types, needs no decoder, and weighs each error by how likely it is to make the circuits that use the block fail.
The tools come from `qldpc.experimental`, so their interface may still change.

## Setup

Run the next cell once in a fresh notebook environment.
IPython's `%pip` magic installs into the active kernel, so the following imports use the same environment.

In [1]:
%%capture
%pip install qldpc

In [2]:
import stim

from qldpc import circuits, codes, decoders
from qldpc.experimental.circuits import (
    ConsumerProfile,
    estimate_downstream_fault_weight,
    estimate_residual_weight_histogram,
)

## A preparation circuit to benchmark

The next cell defines the circuit to benchmark.
To benchmark a circuit of your own, replace its code, circuit, noise, and post-selection options.
The output qubits that hold the prepared state default to the qubits that the circuit acts on and never measures; `output_qubits` selects them explicitly when other qubits stay unmeasured.
Here the circuit is the flagged preparation of the Steane code's $|0_L\rangle$ state in Figure 1c of [Goto (2016)](https://www.nature.com/articles/srep19578), under circuit-level depolarizing noise at a physical error rate of $p = 10^{-3}$.
Its flag qubit, qubit 7, catches faults of a single error mechanism that would otherwise leave a heavy error.
A repeat-until-success protocol prepares the block again whenever the flag fires, which the post-selection options model: `add_flags=True` adds a detector for the flag measurement, and `post_select="flags"` discards the shots in which it fires.

A code of distance $d$ corrects every error of weight up to $t = \lfloor (d - 1) / 2 \rfloor$, which is $t = 1$ for the Steane code.

In [3]:
code = codes.SteaneCode()
prep_circuit = stim.Circuit("""
    # non-fault-tolerant preparation of the logical zero state
    H 0 2 4
    CX 0 3 2 1 4 5
    CX 0 1 2 6 4 3
    CX 2 5 3 6
    # flag qubit 7 detects errors in a logical Z representative
    H 7
    CZ 7 1 7 3 7 5
    MX 7
""")
error_rate = 1e-3
noise_model = circuits.DepolarizingNoiseModel(error_rate)
noisy_circuit = noise_model.noisy_circuit(prep_circuit)
options = {"add_flags": True, "post_select": "flags"}
correctable_weight = (code.get_distance() - 1) // 2

## The usual benchmarks

The next cell runs both of the usual benchmarks on this circuit.
Its logical error rate decodes with the lookup decoder of the state-preparation notebook and post-selects on the added flags, matching the options above.

In [4]:
certificate = circuits.certify_bounded_fault_propagation(
    noisy_circuit, max_weight=correctable_weight, **options
)
print(f"bounded fault propagation up to fault weight {correctable_weight}: {certificate.certified}")

num_samples = 4 * 10**7  # fewer samples give a quicker, noisier estimate


def get_logical_error_rate(circuit: stim.Circuit) -> float:
    # sample the logical error rate of a noisy preparation circuit, post-selecting on its flags;
    # the code's qubits are taken to be qubits 0, 1, ..., len(code) - 1 of the circuit
    diagnostic_circuit, detector_record = circuits.get_state_prep_diagnostic_circuit(
        code, circuit, add_flags=True
    )
    logical_error_rate, _ = circuits.get_logical_error_and_discard_rate(
        diagnostic_circuit,
        decoders.lookup(max_weight=correctable_weight),
        num_samples=num_samples,
        post_select=detector_record.get_events("flags"),
    )
    return logical_error_rate


print(f"logical error rate: {get_logical_error_rate(noisy_circuit):.1e}")

bounded fault propagation up to fault weight 1: True
logical error rate: 1.4e-05


Within the noise model of the circuit, the certificate is exact: every fault of at most $t$ error mechanisms that passes post-selection leaves an error no heavier than its number of error mechanisms, up to stabilizers of the prepared state.
The logical error rate is a sampled estimate, with a statistical uncertainty of about five percent here.
Neither number says how often the block carries a Z-type error, or a correctable error, into the circuits that use it.

## Residual weights

In each shot of the noisy circuit, some of its error mechanisms occur together, and this set of error mechanisms, called a fault, leaves a Pauli error on the prepared block.
Multiplying that Pauli error by a stabilizer of the prepared state does not change the state, so what matters is the smallest weight among all of these equivalent Pauli errors.
This smallest weight is the residual weight of the fault.
For a CSS state such as this one, the weight is by default the CSS weight $\max(|x|, |z|)$ of a Pauli error with X support $x$ and Z support $z$; `css_weight=False` selects the number of qubits that it acts on instead.
A residual weight of zero means that the shot prepared the ideal state, and a residual weight above $t$ lies outside the radius within which the code is guaranteed to correct errors.

`estimate_residual_weight_histogram` estimates the distribution of the residual weight over the shots that pass post-selection.
It computes the contributions of faults with few error mechanisms exactly and samples the rarer faults with more error mechanisms, so its results are estimates with first-order standard errors.

In [5]:
histogram = estimate_residual_weight_histogram(
    noisy_circuit, max_residual_weight=correctable_weight, seed=0, **options
)
print(f"probability that post-selection keeps a shot: {histogram.acceptance_probability:.4f}")
for weight, probability in enumerate(histogram.get_residual_weight_distribution()):
    label = f"{weight}" if weight <= correctable_weight else f"> {correctable_weight}"
    print(f"  residual weight {label}: {probability:.2e}")

probability that post-selection keeps a shot: 0.9925
  residual weight 0: 9.94e-01
  residual weight 1: 6.33e-03
  residual weight > 1: 1.05e-05


Post-selection keeps an estimated 99.25% of shots, so about 0.75% of preparations are repeated.
Most kept shots prepare the ideal state, and about six in a thousand leave an error of residual weight one, which the decoder above corrects in most cases, so it seldom registers as a logical failure.
A residual weight above $t$ is hundreds of times rarer, because the certificate rules it out for every kept fault of a single error mechanism.

## A single figure of merit

How much a residual error matters depends on the circuit that uses the block next, which we call its consumer, such as a round of fault-tolerant error correction.
A fault-tolerant consumer is designed to correct every combination of errors whose total weight is at most $t$, with its own faults counting against that budget too.
We therefore use a deliberately pessimistic model of the consumer, in which a block of residual weight $m$ survives a consumer with $g$ occurring error mechanisms exactly when $m + g \le t$.
Assuming that the faults of the preparation and of the consumer are independent, the figure of merit is the probability that the block's residual error causes a failure that a perfect block would have avoided:

$$
\Phi = \Pr[m + g > t] - \Pr[g > t].
$$

A real decoder corrects many heavier errors as well, so $\Phi$ measures how often the residual error breaks this guarantee, rather than predicting how often a real consumer fails.

If the consumer has many independent error mechanisms, each with a small probability, $g$ is approximately Poisson-distributed with a mean $\nu$, the expected fault weight of the consumer.
`estimate_downstream_fault_weight` estimates $\nu$ by counting the operations of a reference consumer, by default a round of Steane-type error correction: it couples the block to fresh code blocks through $2n$ transversal two-qubit gates and measures $2n$ qubits, so $\nu \approx 2n(p_2 + p_m)$, where $n$ is the number of qubits in the block and $p_2$ and $p_m$ are the error probabilities of a two-qubit gate and a measurement.

In [6]:
expected_fault_weight = estimate_downstream_fault_weight(noisy_circuit, code=code)
consumer = ConsumerProfile.poisson(expected_fault_weight, correctable_weight)
figure_of_merit, standard_error = histogram.get_excess_failure_probability(consumer)
print(f"expected fault weight of the consumer: nu = {expected_fault_weight:.3f}")
print(
    f"figure of merit: Phi = {figure_of_merit:.2e} (sampling standard error {standard_error:.0e})"
)

fault_free_consumer = ConsumerProfile.hard(correctable_weight)
residual_tail, _ = histogram.get_excess_failure_probability(fault_free_consumer)
print(f"with a fault-free consumer (nu = 0): Phi = Pr[m > t] = {residual_tail:.2e}")

expected fault weight of the consumer: nu = 0.028
figure of merit: Phi = 1.83e-04 (sampling standard error 5e-14)
with a fault-free consumer (nu = 0): Phi = Pr[m > t] = 1.05e-05


Here $\nu = 0.028$ estimates the expected number of error mechanisms that occur in one round of the reference consumer; it is not a failure probability.
With this consumer, $\Phi$ is more than ten times larger than with a fault-free consumer, because a residual error of weight one together with a single error mechanism of the consumer already exceeds the budget $t = 1$.
For this circuit, $\Phi$ is therefore dominated by how often the block carries any residual error at all, which a logical error rate does not measure.
The sampling standard error is negligible here because the faults that dominate $\Phi$ are listed exactly; it does not include the uncertainty of the consumer model or of neglected rare faults.

## Errors that a logical error rate cannot see

To compare the two benchmarks directly, the next cell adds Z-type noise to the code's qubits after the preparation, as an idle period might, and recomputes the logical error rate and $\Phi$.

In [7]:
z_idling = stim.Circuit(f"Z_ERROR({error_rate}) " + " ".join(map(str, range(len(code)))))
idle_circuit = noisy_circuit + z_idling
idle_histogram = estimate_residual_weight_histogram(
    idle_circuit, max_residual_weight=correctable_weight, seed=0, **options
)
idle_figure_of_merit, _ = idle_histogram.get_excess_failure_probability(consumer)
for label, block_histogram in [("without", histogram), ("with", idle_histogram)]:
    residual_error = 1 - block_histogram.get_residual_weight_distribution()[0]
    print(f"{label} Z idling: probability of a residual error {residual_error:.2e}")
print(f"with Z idling: logical error rate {get_logical_error_rate(idle_circuit):.1e}")
print(f"with Z idling: Phi = {idle_figure_of_merit:.2e}")

without Z idling: probability of a residual error 6.34e-03
with Z idling: probability of a residual error 1.33e-02
with Z idling: logical error rate 1.8e-05
with Z idling: Phi = 3.71e-04


The idle noise about doubles the probability that the block carries a residual error into its consumer, and $\Phi$ about doubles with it.
The logical error rate also rises, through the decoder confusion described in the introduction: the lookup table explains a syndrome with at most one error mechanism, so an idle Z-type error in the same shot as a correctable error that flips $Z_L$ can make it mispredict $Z_L$.
A decoder that corrected X-type errors from the Z-type stabilizers alone would not be affected, while $\Phi$ counts the Z-type errors directly through the residual weight, whatever the decoder.
For the Steane code, every Z-type error on $|0_L\rangle$ is equivalent to one of weight at most one, so this noise matters only in combination with other errors, which $\Phi$ accounts for through the consumer.

## Comparing preparation protocols

$\Phi$ is defined on the shots that pass post-selection, so it should be read together with the probability of keeping a shot, which sets how often a preparation must be repeated.
The next cell compares the flagged protocol with the same circuit without post-selection on its flag.

In [8]:
for post_select in ["flags", False]:
    protocol_options = {**options, "post_select": post_select}
    certified = circuits.certify_bounded_fault_propagation(
        noisy_circuit, max_weight=correctable_weight, **protocol_options
    ).certified
    protocol_histogram = estimate_residual_weight_histogram(
        noisy_circuit, max_residual_weight=correctable_weight, seed=0, **protocol_options
    )
    protocol_figure_of_merit, _ = protocol_histogram.get_excess_failure_probability(consumer)
    print(
        f"post_select={post_select!r}: certified {certified}, "
        f"keeps {protocol_histogram.acceptance_probability:.4f} of shots, "
        f"Phi {protocol_figure_of_merit:.2e}"
    )

post_select='flags': certified True, keeps 0.9925 of shots, Phi 1.83e-04
post_select=False: certified False, keeps 1.0000 of shots, Phi 2.14e-03


Without post-selection, the circuit is not certified, and the estimated $\Phi$ is more than ten times larger, in exchange for never repeating a preparation.
Such pairs of $\Phi$ and acceptance probability let one compare candidate circuits and post-selection rules on both axes.

## Adapting this notebook

To benchmark another preparation, replace the code, the circuit, the noise model, and the post-selection options in the cell that defines them.
The circuit must be a noisy Stim Clifford circuit that starts in Stim's all-zero state and leaves its output qubits in a stabilizer state.
`post_select` accepts the same options as `circuits.certify_bounded_fault_propagation`, including detectors already present in the circuit.
The helper for the logical error rate and the idle noise both take the code's qubits to be the first `len(code)` qubits of the circuit, and the helper post-selects on added flags, so adjust them for other choices.
`max_residual_weight` sets the largest residual weight that the histogram resolves; use at least $t$, and more to score consumers that resolve heavier residual errors.
The default $\nu$ reads error probabilities from the noisy circuit and counts the operations of a round of Steane-type error correction.
For another consumer, pass its own expected fault weight to `ConsumerProfile.poisson`, or its distribution of fault weights to `ConsumerProfile.from_downstream_fault_weights`, which can also include the residual weights of other prepared blocks that it uses.
A consumer profile can also be built from measured failure probabilities of a real consumer and decoder, which replaces the pessimistic model above.

Within the noise model of the circuit, the certificate is exact for faults of at most $t$ error mechanisms.
The residual-weight histogram is exact for faults of few error mechanisms and sampled for the rest; its first-order standard errors cover only that sampling, and they can be optimistic when the outcomes that matter are rare among the samples.
The logical error rates are sampled estimates.
$\Phi$ further assumes independent faults in the preparation and the consumer, a consumer that fails exactly when the total weight exceeds $t$, a Poisson-distributed fault weight for the consumer, and a value of $\nu$ that counts only the gates and measurements of the reference consumer.
It is therefore a figure of merit for comparing circuits, not a predicted logical error rate.
The [API reference](https://qldpc.readthedocs.io/autoapi/qldpc/experimental/circuits/benchmarking/index.html) defines these quantities precisely and states the remaining limitations.